# Fundamentals of Data Science — Case Study 1
## Market Basket and Transaction Clustering Analysis

**Dataset:** Online retail transaction data  
**Goal:** Explore transaction patterns, represent baskets numerically, reduce dimensions with PCA, and compare four clustering approaches.

> This is an alternative implementation of the uploaded notebook. It uses a different data-loading/preprocessing approach, includes validation, avoids the undefined `X_pca_sub` variable, and handles DBSCAN noise carefully. Enter your own name and roll number if required.


## 1. Import libraries

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn openpyxl

import io, os, zipfile, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
OUTPUT_DIR = "/content/case_study_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Upload and read the transaction file

Upload the original `Online retail_new.csv` file. Each row is interpreted as one basket and comma-separated values are interpreted as products. Reading with `header=None` prevents the first basket from being mistaken for column names when the source is a conventional basket-per-row file.


In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Upload a CSV transaction file to continue.")

source_file = csv_files[0]
raw_df = pd.read_csv(source_file, header=None, dtype=str, keep_default_na=False)
print("File:", source_file)
print("Raw shape:", raw_df.shape)
display(raw_df.head())


In [ ]:
def normalize_product(value):
    text = str(value).strip().lower()
    return "" if text in {"", "nan", "none", "null"} else text

basket_list = []
for row in raw_df.itertuples(index=False, name=None):
    products = [normalize_product(v) for v in row]
    products = [p for p in products if p]
    # Keep each product once per transaction for presence-based analysis.
    basket_list.append(sorted(set(products)))

empty_count = sum(not basket for basket in basket_list)
basket_list = [basket for basket in basket_list if basket]

print(f"Valid transactions: {len(basket_list):,}")
print(f"Empty rows removed: {empty_count:,}")
print(f"Example transaction: {basket_list[0] if basket_list else 'No transactions found'}")
if not basket_list:
    raise ValueError("No valid baskets were found. Check the uploaded CSV format.")


## 3. Descriptive statistics and product frequencies

In [ ]:
basket_length = pd.Series([len(basket) for basket in basket_list], name="items_per_basket")
counts = Counter(product for basket in basket_list for product in basket)
frequency_df = (pd.DataFrame(counts.items(), columns=["Product", "Transaction_Count"])
                .sort_values(["Transaction_Count", "Product"], ascending=[False, True])
                .reset_index(drop=True))
frequency_df["Support"] = frequency_df["Transaction_Count"] / len(basket_list)

print("Number of distinct products:", len(counts))
print(f"Mean basket size: {basket_length.mean():.2f}")
print(f"Median basket size: {basket_length.median():.1f}")
print("\nBasket-size summary:")
display(basket_length.describe().to_frame())
print("\nTop 15 products:")
display(frequency_df.head(15))


In [ ]:
# Figure 1: top products
plot_data = frequency_df.head(15).sort_values("Transaction_Count")
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_data, x="Transaction_Count", y="Product", color="steelblue")
plt.title("Figure 1. Most Frequently Purchased Products")
plt.xlabel("Number of transactions containing product")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure1_top_products.png"), dpi=180)
plt.show()

# Figure 2: basket sizes
plt.figure(figsize=(9, 5))
sns.histplot(basket_length, discrete=True, bins=np.arange(0.5, min(25, basket_length.max()+1.5), 1),
             color="darkcyan", edgecolor="white")
plt.title("Figure 2. Transaction Size Distribution")
plt.xlabel("Distinct products in basket")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure2_basket_sizes.png"), dpi=180)
plt.show()


## 4. Encode baskets and apply PCA

A binary feature indicates whether a product occurs in a basket. Clustering uses the binary features, while PCA provides a 2D view. To keep runtime manageable, PCA and the clustering comparison use the same reproducible sample of up to 2,500 baskets.


In [ ]:
encoder = MultiLabelBinarizer()
basket_matrix = encoder.fit_transform(basket_list).astype(np.uint8)
print("Transaction-product matrix:", basket_matrix.shape)

rng = np.random.default_rng(SEED)
n_sample = min(2500, basket_matrix.shape[0])
sample_ids = np.sort(rng.choice(basket_matrix.shape[0], size=n_sample, replace=False))
X = basket_matrix[sample_ids].astype(np.float32)

pca_model = PCA(n_components=2, random_state=SEED)
X_pca = pca_model.fit_transform(X)
pc1, pc2 = pca_model.explained_variance_ratio_
print(f"PC1 variance explained: {pc1*100:.2f}%")
print(f"PC2 variance explained: {pc2*100:.2f}%")
print(f"Total variance in 2 PCs: {(pc1+pc2)*100:.2f}%")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=13, alpha=0.55)
plt.title("Figure 3. PCA Projection of Transaction Baskets")
plt.xlabel(f"Principal Component 1 ({pc1*100:.1f}%)")
plt.ylabel(f"Principal Component 2 ({pc2*100:.1f}%)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure3_pca_projection.png"), dpi=180)
plt.show()


## 5. Run four clustering algorithms

The four models use the same sample. DBSCAN labels noise observations as `-1`; its scores are calculated only on non-noise observations and should not be compared without also considering how many baskets it excludes.


In [ ]:
cluster_labels = {}
models = {
    "K-Means (k=4)": KMeans(n_clusters=4, random_state=SEED, n_init=10),
    "Hierarchical (Ward, k=4)": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "DBSCAN (eps=1.5)": DBSCAN(eps=1.5, min_samples=15),
    "Spectral (k=4)": SpectralClustering(
        n_clusters=4, affinity="nearest_neighbors", n_neighbors=10,
        assign_labels="kmeans", random_state=SEED
    )
}

metric_records = []
for model_name, model in models.items():
    labels = model.fit_predict(X)
    cluster_labels[model_name] = labels

    non_noise = labels != -1
    valid_labels = labels[non_noise]
    valid_X = X[non_noise]
    distinct_clusters = np.unique(valid_labels)
    sil_value, db_value = np.nan, np.nan
    if len(distinct_clusters) >= 2 and len(distinct_clusters) < len(valid_labels):
        sil_value = silhouette_score(valid_X, valid_labels)
        db_value = davies_bouldin_score(valid_X, valid_labels)

    cluster_sizes = pd.Series(valid_labels).value_counts()
    metric_records.append({
        "Algorithm": model_name,
        "Clusters_excluding_noise": len(distinct_clusters),
        "Noise_count": int((labels == -1).sum()),
        "Noise_percent": round(100 * (labels == -1).mean(), 2),
        "Silhouette_excluding_noise": sil_value,
        "Davies_Bouldin_excluding_noise": db_value,
        "Largest_cluster": int(cluster_sizes.max()) if not cluster_sizes.empty else 0
    })

metrics_df = pd.DataFrame(metric_records)
display(metrics_df)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "clustering_metrics.csv"), index=False)


## 6. Visualize and compare cluster assignments

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (model_name, labels) in zip(axes.ravel(), cluster_labels.items()):
    points = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", s=14, alpha=0.7)
    ax.set_title(model_name)
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
fig.suptitle("Figure 4. Clustering Algorithms Compared in PCA Space", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure4_cluster_comparison.png"), dpi=180)
plt.show()


## 7. Interpret K-means clusters

In [ ]:
# Profile each K-means cluster using the product-presence rate within that cluster.
km_labels = cluster_labels["K-Means (k=4)"]
profile_records = []
product_names = np.array(encoder.classes_)

for cluster_id in sorted(np.unique(km_labels)):
    member_mask = km_labels == cluster_id
    prevalence = X[member_mask].mean(axis=0)
    best_cols = np.argsort(prevalence)[::-1][:10]
    for col in best_cols:
        profile_records.append({
            "Cluster": int(cluster_id),
            "Cluster_Size": int(member_mask.sum()),
            "Product": product_names[col],
            "Within_Cluster_Support": float(prevalence[col])
        })

profiles_df = pd.DataFrame(profile_records)
display(profiles_df)
profiles_df.to_csv(os.path.join(OUTPUT_DIR, "kmeans_cluster_profiles.csv"), index=False)


## 8. Export tables and figures

In [ ]:
# Export the normalized transaction data and product counts.
pd.DataFrame({
    "Transaction_ID": np.arange(1, len(basket_list) + 1),
    "Basket_Size": [len(basket) for basket in basket_list],
    "Products": [", ".join(basket) for basket in basket_list]
}).to_csv(os.path.join(OUTPUT_DIR, "cleaned_transactions.csv"), index=False)

frequency_df.to_csv(os.path.join(OUTPUT_DIR, "product_frequencies.csv"), index=False)

zip_file = "/content/Case_Study_1_Alternative_Results.zip"
with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder, _, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            full_path = os.path.join(folder, filename)
            archive.write(full_path, arcname=os.path.relpath(full_path, OUTPUT_DIR))

print("Output files saved in:", OUTPUT_DIR)
print("Results archive:", zip_file)
print("\nInterpretation guidance:")
print("- Higher silhouette is generally better, but only for comparable samples and scoring rules.")
print("- Lower Davies–Bouldin is generally better.")
print("- For DBSCAN, always report the noise percentage; a high score on a small subset may be misleading.")
print("- PCA plots show only two dimensions and may hide structure in the full feature space.")


In [ ]:
from google.colab import files
files.download("/content/Case_Study_1_Alternative_Results.zip")
